<a href="https://colab.research.google.com/github/ALbahrani5/ML-PROJECT/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
Unit of Analysis: One row represents one unique URL / Content Page (content_id) aggregated at monthly granularity.
Time Window: Mid-panel observation month is month = '2026-03' (March 2026).
Verification Claim: Each content_id appears exactly once for the chosen month, with zero duplicates.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
Features (Input signals):

content_age_days (Age of the page)

impressions_historical (Historical organic impressions)

clicks_28d (Clicks in the last 28 days)

position_drift (Changes in average position)

Label / Target (Proxy):

traffic_decay_pct (Percentage drop in impressions comparing current period vs historical benchmark)

Context (Metadata / Keys):

content_id, client_hash_id, month

Excluded Fields (and why):

clicks_future_month, impressions_future_month — Why: Excluded deliberately to prevent Data Leakage (using information from the future that would not be available at inference time).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
import duckdb
import glob

# 1. Target performance partition files
all_parquet_files = glob.glob(f"{local_repo_dir}/**/*.parquet", recursive=True)
target_files = [f for f in all_parquet_files if "fact_content_daily_performance" in f and "sample" not in f]

con = duckdb.connect()

# 2. Combined Grain & Availability Check for March 2026
query_validation = f"""
SELECT
    -- Grain Validation Metrics
    COUNT(*) as total_rows,
    COUNT(DISTINCT (content_hash_id, report_date)) as unique_daily_records,
    COUNT(DISTINCT content_hash_id) as unique_content_ids,

    -- Availability Filter Metrics
    COUNT(*) FILTER (WHERE gsc_data_available) as gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available) as ga4_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available AND ga4_data_available) as both_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available OR ga4_data_available) as either_available_rows
FROM read_parquet({target_files})
WHERE month = '2026-03';
"""

df_val = con.execute(query_validation).df()
print(df_val.T)  # Transposed print for clean vertical reading

# 3. Final Grain Verdict
is_grain_valid = df_val['total_rows'][0] == df_val['unique_daily_records'][0]
print(f"\nDaily Grain Check Passed (1 row per content_hash_id per report_date): {is_grain_valid}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                             0
total_rows             9841378
unique_daily_records   9841378
unique_content_ids      331437
gsc_available_rows     3611061
ga4_available_rows      413966
both_available_rows     364347
either_available_rows  3660680

Daily Grain Check Passed (1 row per content_hash_id per report_date): True


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
GSC-Only Early Data: Historical performance before recent tracking relies solely on Google Search Console metrics, which may omit off-page context.

Unbalanced History for New URLs: Pages with less than 30 days of lifetime (content_age_days < 30) exhibit noise in calculating traffic decay due to lack of baseline metrics.

Window Overlaps: Rolling 28-day impression windows close to month boundaries may cause slight feature correlation across consecutive months.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.